## LANDSLIDE MONITORING SCENARIO PIPELINE

In [ ]:
import digitalhub as dh
PROJECT_NAME = "20260713-prova"
proj = dh.get_or_create_project(PROJECT_NAME)

ConnectionError: HTTPSConnectionPool(host='core.rsde.atlas.fbk.eu', port=443): Max retries exceeded with url: /api/v1/projects/datasets (Caused by NameResolutionError("HTTPSConnection(host='core.rsde.atlas.fbk.eu', port=443): Failed to resolve 'core.rsde.atlas.fbk.eu' ([Errno 11001] getaddrinfo failed)"))

### Log artifact

Log the shape file 'Shapes_TN' which can be downloaded from the [WebGIS Portal](https://webgis.provincia.tn.it/) from https://siatservices.provincia.tn.it/idt/vector/p_TN_377793f1-1094-4e81-810e-403897418b23.zip. Unzip the files in a folder named 'Shapes_TN' and then log it


In [ ]:
artifact_name='Shapes_TN'
src_path='Shapes_TN'
artifact_data = proj.log_artifact(name=artifact_name, kind="artifact", source=src_path)

Note that to invoke the operation on the platform, the data should be available as an artifact on the platform datalake.


In [ ]:
artifact = proj.get_artifact("Shapes_TN")
artifact.key

The resulting datasets will be registered as the project artifact in the datalake under the name 'Shapes_TN'

Log the Map aritfact with three files (trentino_slope_map.tiff, trentino_aspect_map.tiff, and legend.qml). The files can be downloaded from the <a href="https://huggingface.co/datasets/lbergamasco/trentino-slope-map/tree/main">Huggingface repository</a>. Copy the three files inside a folder 'Map' and log it as project artifact

In [ ]:
artifact_name='Map'
src_path='Map'
artifact_data = proj.log_artifact(name=artifact_name, kind="artifact", source=src_path)

Check if the artifact is created successfully.

In [ ]:
artifact = proj.get_artifact("Map")
artifact.key

### Elaboration

In [ ]:
function_rs = proj.new_function("insar_preprocessing",kind="container", image="ghcr.io/tn-aixpa/rs-landslide-monitoring:0.15", command="python", code_src="./inference/job_insar_preprocessing.py")

In [2]:
function_rs = proj.get_function("insar_preprocessing")

In [ ]:
run = function_rs.run(
    action="job",
    args=['/shared/job_insar_preprocessing.py', 's1_ascending_orbit_117_2021-01-01_2021-04-30', 's1_descending_orbit_168_2021-01-01_2021-04-30', 'Shapes_TN', 'ammprv_v.shp'],
    fs_group='8877',
    resources={"mem":"2Gi","cpu":"2"},
    volumes=[{
            "volume_type": "persistent_volume_claim",
            "name": "volume-flood",
            "mount_path": "/app/data",
            "spec": {
                "size": "10Gi"
            }}]
    )